# MaskGen Token Regret Critic (Single-Notebook DDP Trainig)


## Setup: Import and Load

In [ ]:
import os
import sys
import torch
import open_clip
import matplotlib.pyplot as plt
from modeling.tatitok import TATiTok
from modeling.maskgen import MaskGen_VQ

# Resolve the repo root robustly so notebook imports and local checkpoints stay aligned.
PROJECT_ROOT_CANDIDATES = [
    os.path.abspath('.'),
    os.path.dirname(os.path.abspath('.')),
    os.path.dirname(os.path.dirname(os.path.abspath('.'))),
]
PROJECT_ROOT = next(
    (path for path in PROJECT_ROOT_CANDIDATES if os.path.isdir(os.path.join(path, 'token_regrate'))),
    os.path.abspath('.'),
)
HF_CACHE_DIR = os.path.abspath(os.environ.get('HF_CACHE_DIR', os.path.join(PROJECT_ROOT, 'hf_cache')))
LOCAL_FILES_ONLY = bool(HF_CACHE_DIR)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR
os.environ['OPENCLIP_CACHE_DIR'] = HF_CACHE_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('HF cache:', HF_CACHE_DIR)
print('LOCAL_FILES_ONLY:', LOCAL_FILES_ONLY)

# Add project root to path for imports
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from token_regrate.config import get_config

# Import from training script - shared training and inference utilities
from token_regrate.train_token_regret_ddp import *

def _resolve_pretrained_path(path_value, cache_root=HF_CACHE_DIR):
    path_value = os.path.expanduser(str(path_value))
    if os.path.isabs(path_value):
        return path_value

    normalized_path = os.path.normpath(path_value)
    cache_relative_path = normalized_path
    cache_prefix = f'hf_cache{os.sep}'
    if normalized_path.startswith(cache_prefix):
        cache_relative_path = normalized_path.split(os.sep, 1)[1]

    candidates = [
        os.path.abspath(normalized_path),
        os.path.abspath(os.path.join(PROJECT_ROOT, normalized_path)),
        os.path.abspath(os.path.join(cache_root, cache_relative_path)),
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]

def load_pretrained_stack(model_cfg=None):
    model_cfg = get_config().model if model_cfg is None else model_cfg
    tok_dir = _resolve_pretrained_path(model_cfg.tokenizer_path)
    gen_dir = _resolve_pretrained_path(model_cfg.generator_path)
    clip_name = str(model_cfg.clip_name)
    clip_pretrained = str(getattr(model_cfg, 'clip_pretrained', 'openai'))
    clip_force_quick_gelu = bool(getattr(model_cfg, 'clip_force_quick_gelu', True))

    print('Loading TA-TiTok tokenizer...')
    tatitok_vq_tokenizer = TATiTok.from_pretrained(pretrained_model_name_or_path=tok_dir, cache_dir=tok_dir)
    tatitok_vq_tokenizer.eval()
    tatitok_vq_tokenizer.requires_grad_(False)

    print('Loading MaskGen-VQ generator...')
    maskgen_vq_generator = MaskGen_VQ.from_pretrained(pretrained_model_name_or_path=gen_dir, cache_dir=gen_dir)
    maskgen_vq_generator.eval()
    maskgen_vq_generator.requires_grad_(False)

    print('Loading CLIP text encoder...')
    clip_encoder, _, _ = open_clip.create_model_and_transforms(
        clip_name, pretrained=clip_pretrained, force_quick_gelu=clip_force_quick_gelu
    )
    del clip_encoder.visual
    clip_tokenizer = open_clip.get_tokenizer(clip_name)
    clip_encoder.transformer.batch_first = False
    clip_encoder.eval()
    clip_encoder.requires_grad_(False)

    print('Pretrained stack ready.')
    return tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder


tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder = load_pretrained_stack()
maskgen_vq_generator = maskgen_vq_generator.to(device)
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(device)
clip_encoder = clip_encoder.to(device)
print('Generator device:', next(maskgen_vq_generator.parameters()).device)
print('Tokenizer device:', next(tatitok_vq_tokenizer.parameters()).device)
print('CLIP text encoder device:', next(clip_encoder.parameters()).device)

## Init all parametres for training and infrence time


In [ ]:
# Load notebook defaults directly from token_regrate/config.py.
CONFIG = get_config()
INF_CFG = CONFIG.inference
TRAIN_CFG = CONFIG.training
DATA_CFG = CONFIG.dataset
EXPERIMENT_CFG = CONFIG.experiment
RUNTIME_CFG = CONFIG.runtime
MODEL = CONFIG.model
NOTEBOOK_GENERATE_KWARGS = {
    'guidance_scale': float(TRAIN_CFG.train_guidance_scale),
    'randomize_temperature': float(TRAIN_CFG.train_randomize_temperature),
    'aesthetic_score': float(TRAIN_CFG.train_aesthetic_score),
    'num_sample_steps': int(MODEL.sample_steps),
    'remask_ratio': float(INF_CFG.remask_ratio),
    'refine_start_step': int(INF_CFG.refine_start_step),
    'margin_threshold': float(TRAIN_CFG.margin_threshold),
    'repair_greedy': bool(INF_CFG.repair_greedy),
}

print('Notebook config loaded from token_regrate/config.py')
print('Inference steps:', MODEL.sample_steps)
print('Training batch size:', TRAIN_CFG.per_gpu_batch_size)
print('Training output dir:', EXPERIMENT_CFG.output_dir)
print('DDP only:', bool(TRAIN_CFG.ddp_only))

## Critic Training Entry Cell

Official mode: train only the critic head from the provided dataset source.


In [ ]:
import subprocess
trained_critic = None

nproc_per_node = torch.cuda.device_count() if torch.cuda.is_available() else 1
ddp_cmd = [
    sys.executable,
    '-m', 'torch.distributed.run',
    f'--nproc_per_node={nproc_per_node}',
    'token_regrate/train_token_regret_ddp.py',
    '--num-epochs', str(TRAIN_CFG.num_epochs),
    '--per-gpu-batch-size', str(TRAIN_CFG.per_gpu_batch_size),
    '--learning-rate', str(TRAIN_CFG.learning_rate),
    '--token-sample-ratio', str(TRAIN_CFG.token_sample_ratio),
    '--lambda-rank', str(TRAIN_CFG.lambda_rank),
    '--rank-margin', str(TRAIN_CFG.rank_margin),
    '--counterfactual-chunk-size', str(TRAIN_CFG.counterfactual_chunk_size),
    '--save-every', str(TRAIN_CFG.save_every),
    '--seed', str(INF_CFG.seed),
    '--output-dir', str(EXPERIMENT_CFG.output_dir),
    '--train-data-source', str(DATA_CFG.source),
    '--train-dataset-mode', str(DATA_CFG.mode),
    '--cc12m-cache-dir', str(DATA_CFG.cc12m_cache_dir),
    '--stream-prefetch-batches', str(DATA_CFG.stream_prefetch_batches),
    '--cc12m-loader-workers', str(DATA_CFG.cc12m_loader_workers),
    '--cc12m-loader-max-pending', str(DATA_CFG.cc12m_loader_max_pending),
]
if bool(DATA_CFG.disable_cc12m_cache):
    ddp_cmd += ['--disable-cc12m-cache']
if str(RUNTIME_CFG.resume_checkpoint) and os.path.isfile(str(RUNTIME_CFG.resume_checkpoint)):
    ddp_cmd += ['--resume-checkpoint', str(RUNTIME_CFG.resume_checkpoint)]

ddp_env = os.environ.copy()
ddp_env['DIST_BACKEND'] = str(TRAIN_CFG.dist_backend)
print('Launching DDP training:', ' '.join(ddp_cmd))
run = subprocess.run(ddp_cmd, check=False, env=ddp_env)
if run.returncode != 0:
    raise RuntimeError(f'DDP training failed with return code {run.returncode}')

## Load Critic Head

In [ ]:
CRITIC_CKPT_PATH = "outputs/token_regret_critic/critic_last.pt"

critic_head = load_trained_critic(CRITIC_CKPT_PATH, maskgen_vq_generator, use_hidden=True)
model_device = next(maskgen_vq_generator.parameters()).device
critic_head = critic_head.to(model_device)
print(f'Loaded critic head from: {CRITIC_CKPT_PATH}')

# Quick comparison: without head (baseline) vs with head (critic-guided) on GenEval Promts
## Critic-Isolated GenEval Check

Use one shared draft token grid for both branches, then verify that the head only changes which positions get remasked by raw top-k critic scores.


In [ ]:
# Critic-isolated GenEval comparison: shared draft tokens + raw top-k verification.
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from token_regrate.train_token_regret_ddp import (
    _compute_cfg_scale,
    _add_gumbel_noise,
    forward_maskgen,
    open_clip_text_encoding,
    prepare_text_guidance,
    remask_positions,
    select_topk_token_positions,
    set_global_seed,
    generate_wrapper,
    load_trained_critic,
)
TRACE_GENEVAL_PROMPT_FILE = os.path.abspath('geneval/prompts/generation_prompts.txt')
TRACE_NUM_PROMPTS = 4
TRACE_START_INDEX = 310
TRACE_REFINE_SOFTMAX_TEMPERATURE = 0.7

if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
MODEL = globals().get('MODEL', CONFIG.model)
RUNTIME_CFG = globals().get('RUNTIME_CFG', CONFIG.runtime)
NOTEBOOK_GENERATE_KWARGS = globals().get(
    'NOTEBOOK_GENERATE_KWARGS',
    {
        'guidance_scale': float(TRAIN_CFG.train_guidance_scale),
        'randomize_temperature': float(TRAIN_CFG.train_randomize_temperature),
        'aesthetic_score': float(TRAIN_CFG.train_aesthetic_score),
        'num_sample_steps': int(MODEL.sample_steps),
        'remask_ratio': float(INF_CFG.remask_ratio),
        'refine_start_step': int(INF_CFG.refine_start_step),
        'margin_threshold': float(TRAIN_CFG.margin_threshold),
        'repair_greedy': bool(INF_CFG.repair_greedy),
    },
)

trace_model_device = next(maskgen_vq_generator.parameters()).device

if 'critic_head' not in globals():
    trace_ckpt_path = globals().get('CRITIC_CKPT_PATH', str(RUNTIME_CFG.resume_checkpoint))
    critic_head = load_trained_critic(trace_ckpt_path, maskgen_vq_generator, use_hidden=True).to(trace_model_device)
critic_head = critic_head.to(trace_model_device)
critic_head.eval()

if not os.path.isfile(TRACE_GENEVAL_PROMPT_FILE):
    raise FileNotFoundError(f'GenEval prompt file not found: {TRACE_GENEVAL_PROMPT_FILE}')

with open(TRACE_GENEVAL_PROMPT_FILE, 'r') as f:
    trace_all_prompts = [line.strip() for line in f if line.strip()]

if len(trace_all_prompts) == 0:
    raise RuntimeError(f'No prompts found in: {TRACE_GENEVAL_PROMPT_FILE}')

trace_start_idx = max(0, int(TRACE_START_INDEX))
trace_end_idx = min(len(trace_all_prompts), trace_start_idx + max(1, int(TRACE_NUM_PROMPTS)))
trace_prompts = trace_all_prompts[trace_start_idx:trace_end_idx]
if len(trace_prompts) == 0:
    raise RuntimeError(
        f'Prompt selection is empty. start={trace_start_idx}, count={TRACE_NUM_PROMPTS}, total={len(trace_all_prompts)}'
    )

def _decode_token_grid(tokens, prompts_local):
    text_guidance = prepare_text_guidance(prompts_local, clip_tokenizer, clip_encoder, trace_model_device)
    decoded = tatitok_vq_tokenizer.decode_tokens(tokens.to(trace_model_device), text_guidance)
    decoded = torch.clamp(decoded, 0.0, 1.0)
    return (decoded * 255.0).permute(0, 2, 3, 1).to('cpu', dtype=torch.uint8).numpy()

@torch.no_grad()
def _refine_from_shared_draft_with_debug(
    draft_ids,
    prompts_local,
    guidance_scale,
    randomize_temperature,
    aesthetic_score,
    num_sample_steps,
    remask_ratio,
    refine_start_step,
    margin_threshold,
    repair_greedy,
    critic_use_hidden=True,
    refine_softmax_temperature=TRACE_REFINE_SOFTMAX_TEMPERATURE,
    softmax_temperature_annealing=True,
    guidance_decay='cosine',
    guidance_decay_scale_pow=1.0,
):
    _ = margin_threshold
    model = maskgen_vq_generator
    critic = critic_head.to(trace_model_device)
    draft_ids = draft_ids.to(trace_model_device)
    num_sample_steps = int(num_sample_steps)
    refine_start_step = int(refine_start_step)
    assert refine_start_step >= 0, 'refine_start_step must be non-negative'
    assert refine_start_step < num_sample_steps, 'refine_start_step must be less than num_sample_steps'

    condition, condition_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, prompts_local)
    none_cond, none_cond_pooled = open_clip_text_encoding(clip_tokenizer, clip_encoder, [''])
    num_samples = condition.shape[0]
    none_cond = none_cond.repeat(num_samples, 1, 1)
    none_cond_pooled = none_cond_pooled.repeat(num_samples, 1, 1)

    ids = draft_ids.clone()
    mask_token_id = int(model.mask_token_id)
    debug = {
        'draft_ids': draft_ids.detach().cpu(),
        'step_records': [],
    }

    for step in range(refine_start_step, num_sample_steps):
        ratio = float(step + 1) / float(num_sample_steps)
        ratio = float(max(1e-6, min(1.0, ratio)))
        annealed_temp = float(randomize_temperature) * (1.0 - ratio)
        is_mask = ids.eq(mask_token_id)

        cfg_scale = _compute_cfg_scale(
            ratio=ratio,
            guidance_scale=guidance_scale,
            guidance_decay=guidance_decay,
            guidance_decay_scale_pow=guidance_decay_scale_pow,
            device=trace_model_device,
        )

        if cfg_scale != 0.0:
            logits_all, hidden_all, text_all = forward_maskgen(
                model,
                torch.cat([ids, ids], dim=0),
                torch.cat([condition, none_cond], dim=0),
                torch.cat([condition_pooled, none_cond_pooled], dim=0),
                sample_aesthetic_score=float(aesthetic_score),
            )
            cond_logits, uncond_logits = logits_all[:num_samples], logits_all[num_samples:]
            logits = cond_logits + (cond_logits - uncond_logits) * cfg_scale
            critic_hidden = hidden_all[:num_samples]
            critic_text = text_all[:num_samples]
        else:
            logits, critic_hidden, critic_text = forward_maskgen(
                model,
                ids,
                condition,
                condition_pooled,
                sample_aesthetic_score=float(aesthetic_score),
            )

        timesteps = torch.full((num_samples,), ratio, device=trace_model_device)
        critic_scores = critic(
            hidden_states=critic_hidden if critic_use_hidden else None,
            logits=logits,
            timesteps=timesteps,
            text_features=critic_text,
        )

        if bool(softmax_temperature_annealing):
            softmax_temperature = 0.5 + 0.8 * (1.0 - ratio)
        else:
            softmax_temperature = max(float(refine_softmax_temperature), 1e-6)
        logits_for_sample = logits / float(softmax_temperature)
        if 0 <= mask_token_id < logits_for_sample.shape[-1]:
            logits_for_sample[..., mask_token_id] = -1e9

        if bool(repair_greedy):
            sampled_ids = logits_for_sample.argmax(dim=-1)
        else:
            sampled_ids = _add_gumbel_noise(logits_for_sample, annealed_temp).argmax(dim=-1)
        sampled_ids = torch.where(is_mask, sampled_ids, ids)

        step_record = {
            'step': int(step),
            'ratio': float(ratio),
            'cfg_scale': float(cfg_scale),
        }

        if step == num_sample_steps - 1:
            step_record['final_step'] = True
            debug['step_records'].append(step_record)
            ids = sampled_ids
            continue

        candidate_mask = sampled_ids.ne(mask_token_id)
        select_idx, select_valid = select_topk_token_positions(
            critic_scores,
            remask_ratio,
            candidate_mask=candidate_mask,
        )
        selected_scores = critic_scores.gather(dim=-1, index=select_idx)
        step_record.update(
            {
                'final_step': False,
                'critic_scores': critic_scores.detach().cpu(),
                'candidate_mask': candidate_mask.detach().cpu(),
                'selected_indices': select_idx.detach().cpu(),
                'selected_valid': select_valid.detach().cpu(),
                'selected_scores': selected_scores.detach().cpu(),
            }
        )
        debug['step_records'].append(step_record)
        ids = remask_positions(sampled_ids, select_idx, mask_token_id, valid_mask=select_valid)

    if ids.eq(mask_token_id).any():
        cfg_scale_last = _compute_cfg_scale(
            ratio=1.0,
            guidance_scale=guidance_scale,
            guidance_decay=guidance_decay,
            guidance_decay_scale_pow=guidance_decay_scale_pow,
            device=trace_model_device,
        )
        if cfg_scale_last != 0.0:
            logits_all_last, _, _ = forward_maskgen(
                model,
                torch.cat([ids, ids], dim=0),
                torch.cat([condition, none_cond], dim=0),
                torch.cat([condition_pooled, none_cond_pooled], dim=0),
                sample_aesthetic_score=float(aesthetic_score),
            )
            cond_logits_last, uncond_logits_last = logits_all_last[:num_samples], logits_all_last[num_samples:]
            logits_last = cond_logits_last + (cond_logits_last - uncond_logits_last) * cfg_scale_last
        else:
            logits_last, _, _ = forward_maskgen(
                model,
                ids,
                condition,
                condition_pooled,
                sample_aesthetic_score=float(aesthetic_score),
            )
        if 0 <= mask_token_id < logits_last.shape[-1]:
            logits_last[..., mask_token_id] = -1e9
        fill_ids = logits_last.argmax(dim=-1)
        ids = torch.where(ids.eq(mask_token_id), fill_ids, ids)

    debug['final_ids'] = ids.detach().cpu()
    return ids, debug

trace_generate_kwargs = dict(NOTEBOOK_GENERATE_KWARGS)
trace_seq_len = None

print(f'Using {len(trace_prompts)} GenEval prompts from index [{trace_start_idx}:{trace_end_idx}]')
print('Shared generator settings across both branches:')
for key in ['guidance_scale', 'randomize_temperature', 'aesthetic_score', 'num_sample_steps', 'remask_ratio', 'refine_start_step', 'repair_greedy']:
    print(f'  {key}={trace_generate_kwargs[key]}')

print('Building one shared draft token grid...')
set_global_seed(int(INF_CFG.seed))
shared_draft_ids = generate_wrapper(
    model=maskgen_vq_generator,
    captions=trace_prompts,
    clip_tokenizer=clip_tokenizer,
    clip_encoder=clip_encoder,
    guidance_scale=float(trace_generate_kwargs['guidance_scale']),
    randomize_temperature=float(trace_generate_kwargs['randomize_temperature']),
    sample_aesthetic_score=float(trace_generate_kwargs['aesthetic_score']),
    num_sample_steps=int(trace_generate_kwargs['num_sample_steps']),
    use_critic_head=False,
    critic=None,
    remask_ratio=float(trace_generate_kwargs['remask_ratio']),
    margin_threshold=float(trace_generate_kwargs['margin_threshold']),
    refine_start_step=int(trace_generate_kwargs['refine_start_step']),
    repair_greedy=bool(trace_generate_kwargs['repair_greedy']),
)

print('Refining from the exact same draft tokens with the critic head...')
critic_tokens, critic_debug = _refine_from_shared_draft_with_debug(
    draft_ids=shared_draft_ids,
    prompts_local=trace_prompts,
    guidance_scale=float(trace_generate_kwargs['guidance_scale']),
    randomize_temperature=float(trace_generate_kwargs['randomize_temperature']),
    aesthetic_score=float(trace_generate_kwargs['aesthetic_score']),
    num_sample_steps=int(trace_generate_kwargs['num_sample_steps']),
    remask_ratio=float(trace_generate_kwargs['remask_ratio']),
    refine_start_step=int(trace_generate_kwargs['refine_start_step']),
    margin_threshold=float(trace_generate_kwargs['margin_threshold']),
    repair_greedy=bool(trace_generate_kwargs['repair_greedy']),
    critic_use_hidden=True,
)

trace_guided_steps = [record for record in critic_debug['step_records'] if not record['final_step']]
if len(trace_guided_steps) == 0:
    raise RuntimeError('No guided remask step is available to inspect. Lower refine_start_step to expose critic-guided remasking.')
first_guided_step = trace_guided_steps[0]
masked_scores = first_guided_step['critic_scores'].masked_fill(~first_guided_step['candidate_mask'], float('-inf'))
recomputed_idx = masked_scores.topk(k=first_guided_step['selected_indices'].shape[1], dim=-1).indices
if not torch.equal(recomputed_idx, first_guided_step['selected_indices']):
    raise RuntimeError('Critic top-k verification failed: selected indices are not the raw top-k critic scores.')

trace_seq_len = int(shared_draft_ids.shape[1])
trace_token_diff = critic_tokens.ne(shared_draft_ids).sum(dim=-1).to('cpu').numpy()
trace_selected_counts = first_guided_step['selected_valid'].sum(dim=-1).numpy()
trace_selected_scores = first_guided_step['selected_scores']

print('Verification passed: the first guided remask uses the raw top-k critic scores on the candidate mask.')
print(f"First guided step: step={first_guided_step['step']}, ratio={first_guided_step['ratio']:.4f}, cfg_scale={first_guided_step['cfg_scale']:.4f}")
for sample_idx in range(len(trace_prompts)):
    valid_positions = first_guided_step['selected_valid'][sample_idx]
    chosen_indices = first_guided_step['selected_indices'][sample_idx][valid_positions].tolist()
    chosen_scores = [round(float(x), 4) for x in trace_selected_scores[sample_idx][valid_positions][:8].tolist()]
    print(f'  sample {sample_idx}: first-step remasked {int(trace_selected_counts[sample_idx])} positions')
    print(f'    indices[:8]={chosen_indices[:8]}')
    print(f'    scores[:8]={chosen_scores}')

trace_draft_images = _decode_token_grid(shared_draft_ids, trace_prompts)
trace_critic_images = _decode_token_grid(critic_tokens, trace_prompts)
trace_abs_diff = np.abs(trace_critic_images.astype(np.int16) - trace_draft_images.astype(np.int16)).astype(np.uint8)

SHARED_DRAFT_TRACE = {
    'prompts': trace_prompts,
    'draft_ids': shared_draft_ids.detach().cpu(),
    'critic_tokens': critic_tokens.detach().cpu(),
    'critic_debug': critic_debug,
}

n = len(trace_prompts)
fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
axes = np.array(axes).reshape(n, 3)

for i in range(n):
    prompt_title = trace_prompts[i].replace('\n', ' ').strip()
    if len(prompt_title) > 100:
        prompt_title = prompt_title[:97] + '...'

    ax0 = axes[i, 0]
    ax0.imshow(trace_draft_images[i])
    ax0.axis('off')
    ax0.set_title(f'Shared Draft | {prompt_title}', fontsize=9)

    ax1 = axes[i, 1]
    ax1.imshow(trace_critic_images[i])
    ax1.axis('off')
    ax1.set_title(f'With Head | changed {int(trace_token_diff[i])}/{trace_seq_len} tokens', fontsize=9)

    ax2 = axes[i, 2]
    ax2.imshow(trace_abs_diff[i])
    ax2.axis('off')
    ax2.set_title('Absolute Pixel Delta', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Checkpoint-vs-checkpoint comparison: token deltas + decoded image deltas
import math
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

# Pick two checkpoints to compare.
CKPT_A = 'outputs/token_regret_critic/critic_step_5.pt'
CKPT_B = 'outputs/token_regret_critic/critic_last.pt'

# Reuse prompt list from previous cell if available; otherwise load fresh prompts.
COMPARE_PROMPT_FILE = os.path.abspath('geneval/prompts/generation_prompts.txt')
COMPARE_NUM_PROMPTS = 2
COMPARE_START_INDEX = 10

# Notebook cells may be run out of order, so rebuild config-backed defaults if needed.
if 'CONFIG' not in globals():
    CONFIG = get_config()
INF_CFG = globals().get('INF_CFG', CONFIG.inference)
TRAIN_CFG = globals().get('TRAIN_CFG', CONFIG.training)
MODEL = globals().get('MODEL', CONFIG.model)

# Keep settings aligned with your visual test cell.
COMPARE_SEED = int(INF_CFG.seed)
COMPARE_NUM_SAMPLE_STEPS = int(MODEL.sample_steps)
COMPARE_REMASK_RATIO = INF_CFG.remask_ratio
COMPARE_REFINE_START_STEP = INF_CFG.refine_start_step
COMPARE_MARGIN_THRESHOLD = TRAIN_CFG.margin_threshold

for p in [CKPT_A, CKPT_B]:
    if not os.path.isfile(p):
        raise FileNotFoundError(f'Checkpoint not found: {p}')

if 'prompts' in globals() and isinstance(prompts, list) and len(prompts) > 0:
    prompts_eval = prompts[:max(1, int(COMPARE_NUM_PROMPTS))]
else:
    if not os.path.isfile(COMPARE_PROMPT_FILE):
        raise FileNotFoundError(f'GenEval prompt file not found: {COMPARE_PROMPT_FILE}')
    with open(COMPARE_PROMPT_FILE, 'r') as f:
        all_prompts = [line.strip() for line in f if line.strip()]
    s = max(0, int(COMPARE_START_INDEX))
    e = min(len(all_prompts), s + max(1, int(COMPARE_NUM_PROMPTS)))
    prompts_eval = all_prompts[s:e]

if len(prompts_eval) == 0:
    raise RuntimeError('No prompts selected for checkpoint comparison.')

model_device = next(maskgen_vq_generator.parameters()).device

critic_a = load_trained_critic(CKPT_A, maskgen_vq_generator, use_hidden=True).to(model_device)
critic_b = load_trained_critic(CKPT_B, maskgen_vq_generator, use_hidden=True).to(model_device)


def _generate_tokens_with_critic(critic):
    set_global_seed(COMPARE_SEED)
    toks = generate_wrapper(
        model=maskgen_vq_generator,
        captions=prompts_eval,
        clip_tokenizer=clip_tokenizer,
        clip_encoder=clip_encoder,
        num_sample_steps=COMPARE_NUM_SAMPLE_STEPS,
        use_critic_head=True,
        critic=critic,
        remask_ratio=COMPARE_REMASK_RATIO,
        refine_start_step=COMPARE_REFINE_START_STEP,
        margin_threshold=COMPARE_MARGIN_THRESHOLD,
        critic_use_hidden=True,
    )
    return toks.to(model_device)


def _decode_tokens(tokens):
    text_guidance = prepare_text_guidance(prompts_eval, clip_tokenizer, clip_encoder, model_device)
    img = tatitok_vq_tokenizer.decode_tokens(tokens, text_guidance)
    img = torch.clamp(img, 0.0, 1.0)
    arr = (img * 255.0).permute(0, 2, 3, 1).to('cpu', dtype=torch.uint8).numpy()
    return arr


print('Comparing checkpoints:')
print(f'  A: {CKPT_A}')
print(f'  B: {CKPT_B}')
print(f'Prompts: {len(prompts_eval)}')

# Run both with same seed for fair token-level comparison.
tokens_a = _generate_tokens_with_critic(critic_a)
tokens_b = _generate_tokens_with_critic(critic_b)

# Token-level stats
token_diff = tokens_a.ne(tokens_b)
changed_tokens = token_diff.sum(dim=-1).to('cpu').numpy()
seq_len = int(tokens_a.shape[1])
changed_ratio = (changed_tokens / float(seq_len)) * 100.0

print()
print('Token differences per sample:')
for i in range(tokens_a.shape[0]):
    print(f'  sample {i}: changed {int(changed_tokens[i])}/{seq_len} ({changed_ratio[i]:.2f}%)')
print(f'Average changed tokens: {float(changed_tokens.mean()):.2f}/{seq_len} ({float(changed_ratio.mean()):.2f}%)')

# Decode and compare images.
imgs_a = _decode_tokens(tokens_a)
imgs_b = _decode_tokens(tokens_b)

abs_diff = np.abs(imgs_a.astype(np.int16) - imgs_b.astype(np.int16)).astype(np.uint8)
img_mae = abs_diff.mean(axis=(1, 2, 3))
img_max = abs_diff.max(axis=(1, 2, 3))

print()
print('Image differences per sample:')
for i in range(len(prompts_eval)):
    print(f'  sample {i}: MAE={img_mae[i]:.2f} px, MAX={int(img_max[i])} px')

# Token-diff map (if token sequence is square) for easy visualization.
side = int(round(math.sqrt(seq_len)))
has_square_grid = (side * side == seq_len)
if has_square_grid:
    token_diff_map = token_diff.to('cpu').numpy().reshape(len(prompts_eval), side, side)
else:
    token_diff_map = None

# Visualize A, B, abs image diff, and token-diff map.
n = len(prompts_eval)
cols = 4 if has_square_grid else 3
fig, axes = plt.subplots(n, cols, figsize=(4.2 * cols, 3.6 * n))
axes = np.array(axes).reshape(n, cols)

for i in range(n):
    p = prompts_eval[i].replace(chr(10), ' ').strip()
    if len(p) > 80:
        p = p[:77] + '...'

    axes[i, 0].imshow(imgs_a[i])
    axes[i, 0].axis('off')
    axes[i, 0].set_title('A: ' + os.path.basename(CKPT_A) + chr(10) + p, fontsize=9)

    axes[i, 1].imshow(imgs_b[i])
    axes[i, 1].axis('off')
    axes[i, 1].set_title('B: ' + os.path.basename(CKPT_B), fontsize=9)

    # Mean-over-channel diff heatmap.
    axes[i, 2].imshow(abs_diff[i].mean(axis=2), cmap='inferno')
    axes[i, 2].axis('off')
    axes[i, 2].set_title(
        '|A-B| heatmap' + chr(10) + f'MAE={img_mae[i]:.2f}, token Δ={changed_ratio[i]:.2f}%',
        fontsize=9,
    )

    if has_square_grid:
        axes[i, 3].imshow(token_diff_map[i], cmap='magma', vmin=0, vmax=1, interpolation='nearest')
        axes[i, 3].axis('off')
        axes[i, 3].set_title(f'Token changed map ({side}x{side})', fontsize=9)

plt.tight_layout()
plt.show()